In [1]:
from natural20.map import Map, Terrain
from natural20.battle import Battle
from natural20.player_character import PlayerCharacter
from natural20.map_renderer import MapRenderer
from natural20.die_roll import DieRoll
from natural20.generic_controller import GenericController
from natural20.session import Session
from natural20.actions.move_action import MoveAction
from natural20.action import Action
from natural20.gym.dndenv import dndenv, action_type_to_int
from gymnasium import register, envs, make
from llm_interface import GPT4Interfacer, LLama3Interface
from natural20.gym.dndenv_controller import DndenvController
from model import QNetwork
from natural20.gym.llm_helpers.prompting_utils import action_to_prompt
from natural20.gym.dndenv import embedding_loader
from natural20.event_manager import EventManager
from natural20.gym.dqn.policy import ModelPolicy
from llm_interface import GPT4Interfacer
import os
import time
import torch
import random
import numpy as np

In [2]:
ROUND_PER_MATCH = 30
LLAMA3_URL = "http://202.92.159.242:8001/v1"
MISTRAL_URL = "http://202.92.159.242:8000/v1"

In [3]:
event_manager = EventManager()
event_manager.standard_cli()
session = Session(root_path="map_with_obstacles", event_manager=event_manager)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
class Agent:
    def action(self, observation, info):
        return random.choice(info['available_moves'])

class CustomAgent(Agent):
    def __init__(self, llm_interface):
        self.llm_interface = llm_interface

    def action(self, observation, info):
        return self.llm_interface.select_action_for_state(observation, info)
    def __str__(self) -> str:
        return "Custom LLM Agent"

class ModelAgent(Agent):
    def __init__(self, model_policy):
        self.model_policy = model_policy

    def action(self, observation, info):
        return self.model_policy.action(observation, info)


Setup the appropriate URLs to your vLLM instances

In [6]:
def prompt_for_variant(variant):
    if variant == "llama3":
        prompt = GPT4Interfacer(debug=True, tools=False, base_url=LLAMA3_URL, api_key="token1234", variant='NousResearch/Meta-Llama-3.1-8B-Instruct')
    elif variant == "gpt4":
        prompt = GPT4Interfacer(debug=True, tools=True, variant='gpt-4o')
    elif variant == "mistral":
        prompt = GPT4Interfacer(debug=True, tools=False, base_url=MISTRAL_URL, api_key="token1234", variant='mistralai/Mistral-7B-Instruct-v0.3')
    else:
        raise ValueError(f"Invalid variant: {variant}")

    return prompt

In [7]:
def start_game(player="rl_rules_trained", adversary="llm_llama3"):
    player_agent = None
    if player == "rl_rules_trained":
        model = ModelPolicy(session, weights_file="model_best_dnd_egreedy.pt", device=device, debug=False)
        player_agent = ModelAgent(model)
    elif player == "rl_llama3_trained":
        model = ModelPolicy(session, weights_file="model_best_llm_adversary.pt", device=device, debug=False)
        player_agent = ModelAgent(model)
    elif player == "rl_mistral_trained":
        model = ModelPolicy(session, weights_file="model_best_llm_adversary_mistral.pt", device=device, debug=False)
        player_agent = ModelAgent(model)
    elif player.startswith("llm"):
        prompt = prompt_for_variant(player.split("_")[1])
        player_agent = CustomAgent(prompt)
    elif player == "random":
        player_agent = Agent()
    else:
        raise ValueError(f"Invalid player: {player}")

    # Setup Adversary
    if adversary == "rl_rules_trained":
        model = ModelPolicy(session, weights_file="model_best_dnd_egreedy.pt", device=device, debug=False)
        adversary_agent = ModelAgent(model)
    elif adversary == "rl_llama3_trained":
        model = ModelPolicy(session, weights_file="model_best_llm_adversary.pt", device=device, debug=False)
        adversary_agent = ModelAgent(model)
    elif adversary == "rl_mistral_trained":
        model = ModelPolicy(session, weights_file="model_best_llm_adversary_mistral.pt", device=device, debug=False)
        adversary_agent = ModelAgent(model)
    elif adversary.startswith("llm"):
        prompt = prompt_for_variant(adversary.split("_")[1])
        adversary_agent = CustomAgent(prompt)
    elif adversary == "ai":
        adversary_agent = None
    elif adversary == "random":
        adversary_agent = Agent()
    else:
        raise ValueError(f"Invalid adversary: {adversary}")

    def reaction_callback(state, reward, done, truncated, info):
        """
        Callback function to be called when the environment is waiting for a reaction from the agent.
        Reactions in DnD are typically reactions to enemy actions, such as opportunity attacks.
        """
        print(f"{info['reactor']}: Reaction for {info['trigger']}:")
        action = player_agent.action(state, info)
        return action

    env = make("dndenv-v0", root_path="map_with_obstacles",
           render_mode="ansi",
           custom_agent=adversary_agent,
           profiles=lambda: random.choice(['high_elf_fighter','halfling_rogue','high_elf_mage']),
                enemies=lambda: random.choice(['high_elf_fighter','halfling_rogue','high_elf_mage']),
                map_file=lambda: random.choice(['maps/simple_map',\
                                                'maps/complex_map', \
                                                'maps/game_map', \
                                                'maps/walled_map']),
           debug=False,
           show_logs=False)
    wins = 0
    losses = 0
    ties = 0
    total_rounds = []
    for round in range(ROUND_PER_MATCH):
        print(f"Round {round + 1}")

        observation, info = env.reset(reaction_callback=reaction_callback)
        action = player_agent.action(observation, info)
        terminal = False
        steps = 0

        while not terminal and steps < 512:
            steps += 1
            observation, reward, terminal, truncated, info = env.step(action)

            if not terminal and not truncated:
                action = player_agent.action(observation, info)

            if terminal or truncated:
                print(f"Reward: {reward}")
                if reward == 10:
                    wins += 1
                elif reward == -10:
                    losses += 1
                else:
                    ties += 1
                break
        total_rounds.append(info['round'])

    print(f"Wins: {wins}, Losses: {losses}, Ties: {ties}")
    return wins, losses, ties, np.mean(total_rounds)

In [8]:
players = ["random", "llm_mistral", "llm_llama3", "rl_rules_trained", "rl_llama3_trained", "rl_mistral_trained"]
adversaries = ["random", "llm_mistral", "llm_llama3", "ai", "rl_rules_trained", "rl_llama3_trained", "rl_mistral_trained"]

match_grid = {}

for player in players:
    for adversary in adversaries:
        if player == adversary:
            continue

        if (adversary, player) in match_grid:
            continue
        try:
            wins, losses, ties, avg_rounds = start_game(player=player, adversary=adversary)
            match_grid[(player, adversary)] = (wins, losses, ties, avg_rounds)
            match_grid[(adversary, player)] = (losses, wins, ties, avg_rounds)
        except Exception as e:
            print(f"Error: {e} on {player} vs {adversary}")
            match_grid[(player, adversary)] = (0, 0, 0, 0)
            match_grid[(adversary, player)] = (0, 0, 0, 0)


Round 1
prompt: -------------------------------
We are playing a game of Dungeons and Dragons 5th Edition. It is current your turn and you play 
as a hero character denoted by P (a level 2 wizard). And you have an enemy donoted by E (a level 2 wizard)which you must defeat. 
Your health is at [100.]% specifically 14/14 
Your Enemies health is at [100.]%
Your current conditions are:
Your enemies current conditions are:
You have the following available actions and movement available:

Available movement: [30]ft
Available actions: 1
Bonus actions: 1
Reactions: 1



Here is a rough sketch of the map that considers line of sight to the enemy.
Here is the map:
____________
_    ...____
_     ..____
_     ..____
_. ***oo____
_.......____
_.....P.____
____________
____________
____________
____________
____________
areas with no characters are represented by a dot (.)
the hero character is represented by a (P)
the enemy character is represented by an (E)
Allies or Party Members are represented 

/home/jedld/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:159: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/home/jedld/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:159: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


prompt: -------------------------------
We are playing a game of Dungeons and Dragons 5th Edition. It is current your turn and you play 
as a hero character denoted by P (a level 2 wizard). And you have an enemy donoted by E (a level 2 rogue)which you must defeat. 
Your health is at [100.]% specifically 14/14 
Your Enemies health is at [100.]%
Your current conditions are:
Your enemies current conditions are:
Currently Dodging
Currently Disengaging
You have the following available actions and movement available:

Available movement: [30]ft
Available actions: 1
Bonus actions: 1
Reactions: 1



Here is a rough sketch of the map that considers line of sight to the enemy.
Here is the map:
____________
____________
____________
____________
____________
____________
......P_____
......._____
......._____
..***oo_____
     .._____
    ..E_____
areas with no characters are represented by a dot (.)
the hero character is represented by a (P)
the enemy character is represented by an (E)
Allies or

/home/jedld/miniconda3/envs/ai322/lib/python3.10/site-packages/torch/nn/modules/conv.py:456: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv2d(input, weight, bias, self.stride,


Reward: 0
Round 2
Reward: -10
Round 3
Reward: -10
Round 4
Reward: -10
Round 5
gomerin: Reaction for opportunity_attack:
Reward: 10
Round 6
Reward: -10
Round 7
Reward: 10
Round 8
Reward: -10
Round 9
Reward: 10
Round 10
Reward: -10
Round 11
Reward: 10
Round 12
Reward: -10
Round 13
Reward: -10
Round 14
Reward: 10
Round 15
Reward: -10
Round 16
Reward: 10
Round 17
Reward: 0
Round 18
Reward: 10
Round 19
Reward: -10
Round 20
Reward: 10
Round 21
Reward: 10
Round 22
Reward: -10
Round 23
Reward: 10
Round 24
Reward: 10
Round 25
Reward: -10
Round 26
Reward: 10
Round 27
Reward: 10
Round 28
Reward: -10
Round 29
gomerin: Reaction for opportunity_attack:
gomerin: Reaction for opportunity_attack:
gomerin: Reaction for opportunity_attack:
gomerin: Reaction for opportunity_attack:
gomerin: Reaction for opportunity_attack:
gomerin: Reaction for opportunity_attack:
gomerin: Reaction for opportunity_attack:
gomerin: Reaction for opportunity_attack:
Reward: 10
Round 30
Reward: 0
Wins: 14, Losses: 13, Ties: 3

Generate Report about the matches

In [9]:
import pandas as pd

# setup a pandas table to plot the wins, losses, and ties for each matchup
df = pd.DataFrame(match_grid).T
df.columns = ['Wins', 'Losses', 'Ties', 'AVG Rounds']
df = df.sort_values('Wins', ascending=False)
df

Wins  Losses  Ties  AVG Rounds
ai                 random              23.0     7.0   0.0    3.200000
                   rl_llama3_trained   21.0     9.0   0.0    5.733333
                   llm_mistral         20.0    10.0   0.0    1.633333
                   rl_mistral_trained  18.0    12.0   0.0    4.833333
llm_mistral        random              18.0    10.0   2.0    9.466667
rl_rules_trained   llm_llama3          18.0    10.0   2.0    6.933333
ai                 llm_llama3          17.0    13.0   0.0    2.200000
rl_rules_trained   ai                  17.0    13.0   0.0    2.900000
rl_llama3_trained  random              17.0     8.0   5.0   15.866667
llm_llama3         llm_mistral         16.0    12.0   2.0    4.566667
                   random              16.0    11.0   3.0   15.333333
rl_mistral_trained random              15.0     9.0   6.0   16.200000
llm_mistral        rl_rules_trained    14.0    11.0   5.0   10.366667
rl_llama3_trained  llm_llama3          14.0    14.0   2.0    8.166667
llm_llama3         rl_llama3_trained   14.0    14.0   2.0    8.166667
                   rl_mistral_trained  14.0    12.0   4.0    8.533333
random             rl_rules_trained    14.0    13.0   3.0   16.400000
rl_llama3_trained  llm_mistral         14.0    13.0   3.0    7.400000
ai                 rl_rules_trained    13.0    17.0   0.0    2.900000
llm_llama3         ai                  13.0    17.0   0.0    2.200000
rl_mistral_trained llm_mistral         13.0    12.0   5.0    8.133333
llm_mistral        rl_llama3_trained   13.0    14.0   3.0    7.400000
rl_rules_trained   random              13.0    14.0   3.0   16.400000
llm_mistral        rl_mistral_trained  12.0    13.0   5.0    8.133333
rl_mistral_trained ai                  12.0    18.0   0.0    4.833333
llm_mistral        llm_llama3          12.0    16.0   2.0    4.566667
rl_rules_trained   rl_mistral_trained  12.0     9.0   9.0   54.500000
rl_mistral_trained llm_llama3          12.0    14.0   4.0    8.533333
rl_rules_trained   llm_mistral         11.0    14.0   5.0   10.366667
random             llm_llama3          11.0    16.0   3.0   15.333333
rl_llama3_trained  rl_rules_trained    11.0     8.0  11.0   64.633333
random             llm_mistral         10.0    18.0   2.0    9.466667
llm_mistral        ai                  10.0    20.0   0.0    1.633333
llm_llama3         rl_rules_trained    10.0    18.0   2.0    6.933333
random             rl_mistral_trained   9.0    15.0   6.0   16.200000
rl_mistral_trained rl_rules_trained     9.0    12.0   9.0   54.500000
rl_llama3_trained  ai                   9.0    21.0   0.0    5.733333
                   rl_mistral_trained   9.0     5.0  16.0   71.200000
rl_rules_trained   rl_llama3_trained    8.0    11.0  11.0   64.633333
random             rl_llama3_trained    8.0    17.0   5.0   15.866667
                   ai                   7.0    23.0   0.0    3.200000
rl_mistral_trained rl_llama3_trained    5.0     9.0  16.0   71.200000

Dump results to CSV

In [10]:
df.to_csv("match_results.csv")

In [11]:
# create a leaderboard on the most wins

leaderboard = df.groupby(level=0).sum().sort_values('Wins', ascending=False)
leaderboard

,Wins,Losses,Ties,AVG Rounds
ai,112.0,68.0,0.0,20.500000
llm_llama3,83.0,84.0,13.0,45.733333
llm_mistral,79.0,84.0,17.0,41.566667
rl_rules_trained,79.0,71.0,30.0,155.733333
rl_llama3_trained,74.0,69.0,37.0,173.000000
rl_mistral_trained,66.0,74.0,40.0,163.400000
random,59.0,102.0,19.0,76.466667


In [12]:
leaderboard.to_csv("leaderboard.csv")